# Amazon Week 5 V4 — PyTorch ListNet One-Click

Run **Runtime → Run all** in Google Colab.

Upload when prompted:
1. `amazon_week5_v1_feature_dataset.zip`
2. `amazon_week5_v2_lambdamart_results.zip`


In [ ]:
# ============================================================
# AMAZON WEEK 5 V4 — PYTORCH LISTNET ONE-CLICK
#
# Upload these 2 ZIP files when prompted:
# 1) amazon_week5_v1_feature_dataset.zip
# 2) amazon_week5_v2_lambdamart_results.zip
#
# Output:
# amazon_week5_v4_listnet_results.zip
#
# What this notebook does:
# - reuses the exact V2 fit/validation query split
# - trains a listwise neural ranker (ListNet)
# - selects among small MLP configs using validation NDCG@10
# - retrains on all 800 training queries
# - evaluates on the untouched 200-query test set
# - compares ListNet with LambdaMART
# ============================================================

!pip -q install "scikit-learn>=1.4" pandas numpy matplotlib pyarrow torch

import os, json, math, shutil, zipfile, warnings, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import ndcg_score
from google.colab import files

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

ROOT = Path("/content/week5_v4")
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# STEP 1 — Upload V1 + V2 ZIPs
# ------------------------------------------------------------
print("=" * 74)
print("STEP 1/8 — Upload Week5 V1 + Week5 V2 ZIPs")
print("=" * 74)

uploaded = files.upload()
names = list(uploaded.keys())

def pick_zip(terms):
    for name in names:
        low = name.lower()
        if name.lower().endswith(".zip") and all(t.lower() in low for t in terms):
            return name
    return None

v1_zip = pick_zip(["week5", "v1"])
v2_zip = pick_zip(["week5", "v2"])

if v1_zip is None or v2_zip is None:
    print("\nUploaded:")
    for x in names:
        print(" -", x)
    raise ValueError(
        "Please upload amazon_week5_v1_feature_dataset.zip and "
        "amazon_week5_v2_lambdamart_results.zip"
    )

print("✓ V1:", v1_zip)
print("✓ V2:", v2_zip)

def extract(name, subdir):
    out = ROOT / subdir
    out.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(name, "r") as z:
        z.extractall(out)
    return out

V1 = extract(v1_zip, "v1")
V2 = extract(v2_zip, "v2")

def find_file(folder, filename):
    hits = list(Path(folder).rglob(filename))
    if not hits:
        hits = [
            p for p in Path(folder).rglob("*")
            if p.is_file() and p.name.lower() == filename.lower()
        ]
    if not hits:
        raise FileNotFoundError(f"Could not find {filename} in {folder}")
    return hits[0]

# ------------------------------------------------------------
# STEP 2 — Load features + exact V2 split
# ------------------------------------------------------------
print("\n" + "=" * 74)
print("STEP 2/8 — Load feature table and V2 validation split")
print("=" * 74)

df = pd.read_parquet(
    find_file(V1, "week5_feature_dataset.parquet")
)

with open(
    find_file(V1, "week5_feature_list.json"),
    "r",
    encoding="utf-8"
) as f:
    feature_cols = json.load(f)

fit_ids = set(
    pd.read_csv(
        find_file(V2, "week5_v2_fit_query_ids.csv")
    )["query_id"].astype(str)
)

val_ids = set(
    pd.read_csv(
        find_file(V2, "week5_v2_validation_query_ids.csv")
    )["query_id"].astype(str)
)

v2_metrics = pd.read_csv(
    find_file(V2, "week5_v2_test_metrics.csv")
)

df["query_id"] = df["query_id"].astype(str)
df["product_id"] = df["product_id"].astype(str)
df["relevance_label"] = pd.to_numeric(
    df["relevance_label"], errors="raise"
).astype(int)

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

train_all = df[df["split"] == "train"].copy()
test_df = df[df["split"] == "test"].copy()

fit_df = train_all[
    train_all["query_id"].isin(fit_ids)
].copy()

val_df = train_all[
    train_all["query_id"].isin(val_ids)
].copy()

print(f"Features         : {len(feature_cols)}")
print(f"Fit queries      : {fit_df['query_id'].nunique()}")
print(f"Validation queries: {val_df['query_id'].nunique()}")
print(f"Final train      : {train_all['query_id'].nunique()}")
print(f"Test queries     : {test_df['query_id'].nunique()}")

# ------------------------------------------------------------
# STEP 3 — Standardize features and build query lists
# ------------------------------------------------------------
print("\n" + "=" * 74)
print("STEP 3/8 — Standardize features and build query groups")
print("=" * 74)

# Fill any numeric NaN using training medians before scaling.
train_medians = fit_df[feature_cols].median(numeric_only=True)
for c in feature_cols:
    fill_value = train_medians.get(c, 0.0)
    if pd.isna(fill_value):
        fill_value = 0.0
    for frame in [fit_df, val_df, train_all, test_df]:
        frame[c] = frame[c].fillna(fill_value)

scaler_tune = StandardScaler()
scaler_tune.fit(fit_df[feature_cols])

def transform_frame(frame, scaler):
    out = frame.copy()
    out.loc[:, feature_cols] = scaler.transform(out[feature_cols])
    return out

fit_scaled = transform_frame(fit_df, scaler_tune)
val_scaled = transform_frame(val_df, scaler_tune)

def build_query_groups(frame):
    groups = []
    for qid, g in frame.groupby("query_id", sort=True):
        X = torch.tensor(
            g[feature_cols].to_numpy(dtype=np.float32),
            dtype=torch.float32
        )
        y = torch.tensor(
            g["relevance_label"].to_numpy(dtype=np.float32),
            dtype=torch.float32
        )
        groups.append((qid, X, y))
    return groups

fit_groups = build_query_groups(fit_scaled)
val_groups = build_query_groups(val_scaled)

print("Fit groups:", len(fit_groups))
print("Validation groups:", len(val_groups))

# ------------------------------------------------------------
# ListNet model + objective
# ------------------------------------------------------------
class ListNetMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(64, 32), dropout=0.10):
        super().__init__()

        layers = []
        prev = input_dim

        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h

        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)

def listnet_loss(scores, labels):
    # ListNet top-one probability distribution.
    # Labels 0–3 are converted to a smooth positive distribution.
    target_prob = F.softmax(labels, dim=0)
    log_pred_prob = F.log_softmax(scores, dim=0)
    return -(target_prob * log_pred_prob).sum()

def predict_groups(model, groups):
    model.eval()
    rows = []

    with torch.no_grad():
        for qid, X, y in groups:
            scores = model(X.to(device)).cpu().numpy()
            labels = y.numpy()

            for i in range(len(scores)):
                rows.append({
                    "query_id": qid,
                    "row_in_query": i,
                    "relevance_label": float(labels[i]),
                    "score": float(scores[i])
                })

    return pd.DataFrame(rows)

def mean_ndcg_from_groups(model, groups, k=10):
    model.eval()
    vals = []

    with torch.no_grad():
        for qid, X, y in groups:
            scores = model(X.to(device)).cpu().numpy()
            labels = y.numpy()

            vals.append(
                ndcg_score(
                    labels.reshape(1,-1),
                    scores.reshape(1,-1),
                    k=k
                )
            )

    return float(np.mean(vals))

def train_listnet(
    fit_groups,
    val_groups,
    hidden_dims,
    dropout,
    lr,
    weight_decay,
    max_epochs=300,
    patience=35
):
    torch.manual_seed(SEED)

    model = ListNetMLP(
        input_dim=len(feature_cols),
        hidden_dims=hidden_dims,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    best_state = None
    best_val = -np.inf
    best_epoch = 0
    no_improve = 0
    history = []

    rng = np.random.RandomState(SEED)

    for epoch in range(1, max_epochs + 1):
        model.train()

        order = rng.permutation(len(fit_groups))
        losses = []

        for idx in order:
            _, X, y = fit_groups[idx]

            X = X.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            scores = model(X)
            loss = listnet_loss(scores, y)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0
            )
            optimizer.step()

            losses.append(float(loss.detach().cpu()))

        val_ndcg = mean_ndcg_from_groups(
            model,
            val_groups,
            k=10
        )

        history.append({
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "val_NDCG@10": val_ndcg
        })

        if val_ndcg > best_val + 1e-6:
            best_val = val_ndcg
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            no_improve = 0
        else:
            no_improve += 1

        if epoch % 25 == 0:
            print(
                f"  epoch={epoch:3d} "
                f"loss={np.mean(losses):.4f} "
                f"val_NDCG@10={val_ndcg:.6f}"
            )

        if no_improve >= patience:
            break

    model.load_state_dict(best_state)

    return (
        model,
        best_val,
        best_epoch,
        pd.DataFrame(history)
    )

# ------------------------------------------------------------
# STEP 4 — Small architecture search
# ------------------------------------------------------------
print("\n" + "=" * 74)
print("STEP 4/8 — Tune ListNet architecture on validation NDCG@10")
print("=" * 74)

configs = [
    {
        "name": "MLP_32",
        "hidden_dims": (32,),
        "dropout": 0.05,
        "lr": 1e-3,
        "weight_decay": 1e-4
    },
    {
        "name": "MLP_64_32",
        "hidden_dims": (64, 32),
        "dropout": 0.10,
        "lr": 1e-3,
        "weight_decay": 1e-4
    },
    {
        "name": "MLP_128_64",
        "hidden_dims": (128, 64),
        "dropout": 0.15,
        "lr": 7e-4,
        "weight_decay": 1e-4
    }
]

search_rows = []
search_histories = {}
best_model = None
best_cfg = None
best_val = -np.inf
best_epoch = None

for cfg in configs:
    print("\nTraining", cfg["name"])

    model, val_ndcg, epoch, history = train_listnet(
        fit_groups,
        val_groups,
        hidden_dims=cfg["hidden_dims"],
        dropout=cfg["dropout"],
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"]
    )

    search_histories[cfg["name"]] = history

    search_rows.append({
        "model": cfg["name"],
        "hidden_dims": str(cfg["hidden_dims"]),
        "dropout": cfg["dropout"],
        "learning_rate": cfg["lr"],
        "weight_decay": cfg["weight_decay"],
        "best_epoch": epoch,
        "validation_NDCG@10": val_ndcg
    })

    print(
        f"✓ {cfg['name']} "
        f"best_epoch={epoch} "
        f"val_NDCG@10={val_ndcg:.6f}"
    )

    if val_ndcg > best_val:
        best_val = val_ndcg
        best_model = model
        best_cfg = cfg.copy()
        best_epoch = int(epoch)

search_df = pd.DataFrame(search_rows).sort_values(
    "validation_NDCG@10",
    ascending=False
).reset_index(drop=True)

print("\nSearch results:")
print(search_df.to_string(index=False))

# ------------------------------------------------------------
# STEP 5 — Retrain chosen architecture on ALL training queries
# ------------------------------------------------------------
print("\n" + "=" * 74)
print("STEP 5/8 — Retrain selected ListNet on all 800 train queries")
print("=" * 74)

# Refit scaler on all train queries.
train_medians_final = train_all[feature_cols].median(numeric_only=True)

for c in feature_cols:
    fill_value = train_medians_final.get(c, 0.0)
    if pd.isna(fill_value):
        fill_value = 0.0
    train_all[c] = train_all[c].fillna(fill_value)
    test_df[c] = test_df[c].fillna(fill_value)

scaler_final = StandardScaler()
scaler_final.fit(train_all[feature_cols])

train_final_scaled = transform_frame(
    train_all,
    scaler_final
)

test_scaled = transform_frame(
    test_df,
    scaler_final
)

train_final_groups = build_query_groups(
    train_final_scaled
)

# For final training we use the validation-selected epoch count.
torch.manual_seed(SEED)

final_model = ListNetMLP(
    input_dim=len(feature_cols),
    hidden_dims=best_cfg["hidden_dims"],
    dropout=best_cfg["dropout"]
).to(device)

optimizer = torch.optim.AdamW(
    final_model.parameters(),
    lr=best_cfg["lr"],
    weight_decay=best_cfg["weight_decay"]
)

rng = np.random.RandomState(SEED)

for epoch in range(1, best_epoch + 1):
    final_model.train()
    order = rng.permutation(len(train_final_groups))
    losses = []

    for idx in order:
        _, X, y = train_final_groups[idx]

        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        scores = final_model(X)
        loss = listnet_loss(scores, y)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            final_model.parameters(),
            max_norm=5.0
        )

        optimizer.step()
        losses.append(float(loss.detach().cpu()))

    if epoch % 25 == 0 or epoch == best_epoch:
        print(
            f"Final train epoch {epoch}/{best_epoch} "
            f"loss={np.mean(losses):.4f}"
        )

# ------------------------------------------------------------
# STEP 6 — Evaluate untouched test set
# ------------------------------------------------------------
print("\n" + "=" * 74)
print("STEP 6/8 — Evaluate ListNet on untouched test queries")
print("=" * 74)

final_model.eval()

# IMPORTANT:
# test_df keeps original row indices from the full dataset, so its indices may
# be larger than len(test_df)-1. Reset to contiguous 0..N-1 positions before
# writing predictions into a NumPy array.
test_out = test_df.copy().reset_index(drop=True)
test_scores = np.zeros(len(test_out), dtype=np.float32)

with torch.no_grad():
    for qid, idxs in test_out.groupby("query_id", sort=False).groups.items():
        idxs = np.asarray(list(idxs), dtype=int)

        X_np = scaler_final.transform(
            test_out.iloc[idxs][feature_cols]
        )

        X = torch.tensor(
            X_np,
            dtype=torch.float32,
            device=device
        )

        s = final_model(X).detach().cpu().numpy()
        test_scores[idxs] = s

test_out["listnet_score"] = test_scores

def evaluate_ranking(frame, score_col):
    nd10, nd3, recalls, mrrs = [], [], [], []

    for _, g in frame.groupby("query_id"):
        y = g["relevance_label"].to_numpy(dtype=float)
        s = g[score_col].to_numpy(dtype=float)

        nd10.append(
            ndcg_score(
                y.reshape(1,-1),
                s.reshape(1,-1),
                k=10
            )
        )

        nd3.append(
            ndcg_score(
                y.reshape(1,-1),
                s.reshape(1,-1),
                k=3
            )
        )

        order = np.argsort(-s, kind="mergesort")
        ranked = y[order]

        total_rel = int((y > 0).sum())
        recalls.append(
            float((ranked[:10] > 0).sum() / total_rel)
            if total_rel > 0 else 0.0
        )

        pos = np.where(ranked > 0)[0]
        mrrs.append(
            float(1.0 / (pos[0] + 1))
            if len(pos) else 0.0
        )

    return {
        "NDCG@10": float(np.mean(nd10)),
        "NDCG@3": float(np.mean(nd3)),
        "Recall@10": float(np.mean(recalls)),
        "MRR": float(np.mean(mrrs)),
        "Queries": int(frame["query_id"].nunique())
    }

listnet_metrics = evaluate_ranking(
    test_out,
    "listnet_score"
)

lm_row = v2_metrics[
    v2_metrics["model"] == "LambdaMART"
].iloc[0]

comparison = pd.DataFrame([
    {
        "model": "LambdaMART",
        "NDCG@10": float(lm_row["NDCG@10"]),
        "NDCG@3": float(lm_row["NDCG@3"]),
        "Recall@10": float(lm_row["Recall@10"]),
        "MRR": float(lm_row["MRR"])
    },
    {
        "model": "ListNet",
        "NDCG@10": listnet_metrics["NDCG@10"],
        "NDCG@3": listnet_metrics["NDCG@3"],
        "Recall@10": listnet_metrics["Recall@10"],
        "MRR": listnet_metrics["MRR"]
    }
])

lm_ndcg = float(lm_row["NDCG@10"])
listnet_ndcg = listnet_metrics["NDCG@10"]

comparison["vs_LambdaMART_NDCG_%"] = (
    (comparison["NDCG@10"] - lm_ndcg)
    / lm_ndcg
    * 100
)

print("\nFINAL COMPARISON")
print(comparison.to_string(index=False))

# ------------------------------------------------------------
# STEP 7 — Save model, outputs, plots
# ------------------------------------------------------------
print("\n" + "=" * 74)
print("STEP 7/8 — Save ListNet outputs")
print("=" * 74)

OUT = ROOT / "output"
OUT.mkdir(parents=True, exist_ok=True)

search_df.to_csv(
    OUT / "week5_v4_listnet_validation_search.csv",
    index=False
)

comparison.to_csv(
    OUT / "week5_v4_listnet_comparison.csv",
    index=False
)

test_out["predicted_rank"] = (
    test_out
    .groupby("query_id")["listnet_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

keep_cols = [
    "query_id",
    "query",
    "product_id",
    "title",
    "relevance_label",
    "listnet_score",
    "predicted_rank"
]
keep_cols = [c for c in keep_cols if c in test_out.columns]

test_out[
    keep_cols
].sort_values(
    ["query_id", "predicted_rank"]
).to_csv(
    OUT / "week5_v4_listnet_test_rankings.csv",
    index=False
)

torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "input_dim": len(feature_cols),
        "hidden_dims": best_cfg["hidden_dims"],
        "dropout": best_cfg["dropout"],
        "feature_cols": feature_cols,
        "best_epoch": best_epoch,
        "scaler_mean": scaler_final.mean_,
        "scaler_scale": scaler_final.scale_
    },
    OUT / "week5_v4_listnet_model.pt"
)

config = {
    "seed": SEED,
    "device": str(device),
    "features": feature_cols,
    "selected_model": best_cfg["name"],
    "hidden_dims": list(best_cfg["hidden_dims"]),
    "dropout": best_cfg["dropout"],
    "learning_rate": best_cfg["lr"],
    "weight_decay": best_cfg["weight_decay"],
    "selected_epoch": best_epoch,
    "validation_NDCG@10": float(best_val),
    "test_NDCG@10": float(listnet_ndcg),
    "LambdaMART_NDCG@10": float(lm_ndcg),
    "ListNet_vs_LambdaMART_pct": float(
        (listnet_ndcg - lm_ndcg) / lm_ndcg * 100
    )
}

with open(
    OUT / "week5_v4_listnet_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(config, f, indent=2)

# Training curve for selected config
selected_history = search_histories[
    best_cfg["name"]
]

plt.figure(figsize=(8,5))
plt.plot(
    selected_history["epoch"],
    selected_history["val_NDCG@10"]
)
plt.xlabel("Epoch")
plt.ylabel("Validation NDCG@10")
plt.title(
    f"Week 5 V4 — ListNet Validation Curve ({best_cfg['name']})"
)
plt.tight_layout()
plt.savefig(
    OUT / "week5_v4_listnet_validation_curve.png",
    dpi=180,
    bbox_inches="tight"
)
plt.close()

# Model comparison plot
plot_df = comparison.sort_values(
    "NDCG@10",
    ascending=True
)

plt.figure(figsize=(7,4.5))
plt.barh(
    plot_df["model"],
    plot_df["NDCG@10"]
)
plt.xlabel("NDCG@10")
plt.ylabel("Model")
plt.title("Week 5 V4 — LambdaMART vs ListNet")
plt.xlim(
    max(0, plot_df["NDCG@10"].min() - 0.04),
    min(1.0, plot_df["NDCG@10"].max() + 0.02)
)
plt.tight_layout()
plt.savefig(
    OUT / "week5_v4_lambdamart_vs_listnet.png",
    dpi=180,
    bbox_inches="tight"
)
plt.close()

readme = f"""
AMAZON SEARCH / LTR — WEEK 5 V4 LISTNET
=======================================

PURPOSE
-------
Implement the Week5 listwise-learning extension using PyTorch ListNet.

DATA SPLIT
----------
The exact same fit / validation query split from Week5 V2 LambdaMART was reused.

Fit queries: {len(fit_ids)}
Validation queries: {len(val_ids)}
Final training queries: {train_all['query_id'].nunique()}
Test queries: {test_df['query_id'].nunique()}

SELECTED LISTNET
----------------
Architecture: {best_cfg['name']}
Hidden dimensions: {best_cfg['hidden_dims']}
Dropout: {best_cfg['dropout']}
Learning rate: {best_cfg['lr']}
Weight decay: {best_cfg['weight_decay']}
Selected epoch: {best_epoch}

Validation NDCG@10:
{best_val:.6f}

TEST RESULTS
------------
ListNet NDCG@10:
{listnet_ndcg:.6f}

LambdaMART NDCG@10:
{lm_ndcg:.6f}

ListNet relative difference vs LambdaMART:
{((listnet_ndcg-lm_ndcg)/lm_ndcg*100):+.3f}%

INTERPRETATION
--------------
ListNet is included as the neural listwise benchmark required by the Week5
roadmap. LambdaMART remains the production-style primary model if it has the
stronger held-out NDCG@10 and better stability on this relatively small
query-level training set.

NEXT
----
Week5 V5:
Final validation + consolidated Week5 report.
""".strip()

with open(
    OUT / "README_WEEK5_V4.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(readme + "\n")

# ------------------------------------------------------------
# STEP 8 — ZIP + download
# ------------------------------------------------------------
print("\n" + "=" * 74)
print("STEP 8/8 — Package ListNet results")
print("=" * 74)

zip_out = Path(
    "/content/amazon_week5_v4_listnet_results.zip"
)

if zip_out.exists():
    zip_out.unlink()

with zipfile.ZipFile(
    zip_out,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p, arcname=p.name)

print("\n✅ WEEK 5 V4 LISTNET COMPLETE")
print("✅ Created:", zip_out)
print("\nUpload the V4 ZIP back to ChatGPT for final Week5 validation/report.")
files.download(str(zip_out))
